#Phase 1: Inspect and Clean
## Step 2: Load the combined dataset
Load all 212 anonymized patients from one file and fix the first column name

In [17]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "phase1_data" else Path.cwd()
path = ROOT / "data" / "raw" / "all_patients.csv"

df = pd.read_csv(path, dtype=str, encoding="utf-8-sig", engine="c", low_memory=False)
df = df.rename(columns={df.columns[0]: "patient"})

funnel = [("Loaded", len(df), df["patient"].nunique())]
print("Rows:", df.shape[0], "| Columns:", df.shape[1], "| Patients:", df["patient"].nunique())

Rows: 117474 | Columns: 115 | Patients: 212


##Step 2: Columns Overview 
Chcek each column's missing %, unique values, and average text length to find useful and useless columns.

In [18]:
overview = pd.DataFrame({
    "missing_%": (df.isna().mean() * 100).round(1),
    "unique": df.nunique(),
    "avg_length": df.apply(lambda s: s.dropna().str.len().mean()).round(1),
}).sort_values("missing_%")

print("Columns 100% empty:", (overview["missing_%"] == 100).sum())
print("Columns with 1 value only:", (overview["unique"] <= 1).sum())
overview.head(30)

Columns 100% empty: 15
Columns with 1 value only: 10


,missing_%,unique,avg_length
patient,0.0,212,4.0
Field2,23.6,81854,22.7
Field3,23.6,1,3.0
Field48,23.6,13142,12.9
Field4,31.2,1,12.0
Field1_Text,40.7,213,4.1
Field87,40.7,211,4.1
Field47,68.9,32650,171.1
Field34,76.4,25216,22.5
Field41,76.4,25213,22.5


## Step 3: Posts per patient
Count how many rows each patient has, to spot patients with too little data.

In [19]:
posts = df["patient"].value_counts()
print("Patients:", posts.size)
print("Rows per patient → min:", posts.min(), "| median:", int(posts.median()), "| max:", posts.max())
print("Patients with fewer than 5 rows:", (posts < 5).sum())

Patients: 212
Rows per patient → min: 19 | median: 583 | max: 2181
Patients with fewer than 5 rows: 0


## Step 4: Drop useless columns
Remove columns that are completely empty or have the same value in every row, since they carry no information.

In [20]:
before = df.shape[1]
empty = [c for c in df.columns if df[c].isna().all()]
constant = [c for c in df.columns if c != "patient" and df[c].nunique() <= 1]
df = df.drop(columns=list(set(empty + constant)))

print("Columns before:", before, "| removed:", before - df.shape[1], "| now:", df.shape[1])

Columns before: 115 | removed: 10 | now: 105


## Step 5: Keep only the patient's own posts
Keep rows where the author is the patient, and drop posts by other people ("OTHER") or with no author, since only the patient's own words show their language.

In [21]:
before = len(df)
df = df[df["Field1_Text"] == df["patient"]]
funnel.append(("Patient's own posts only", len(df), df["patient"].nunique()))

print("Rows before:", before, "| removed:", before - len(df), "| now:", len(df))
print("Patients still present:", df["patient"].nunique())

Rows before: 117474 | removed: 54443 | now: 63031
Patients still present: 212


## Step 6: Find the text columns
Rank the remaining columns by how many rows they fill and their average length, to find where the post text is stored.

In [22]:
meta = {"patient", "Field1_Text", "Field2", "Field87", "Time", "Time1"}
cands = [c for c in df.columns if c not in meta]

text_rank = pd.DataFrame({
    "filled_rows": df[cands].notna().sum(),
    "avg_length": df[cands].apply(lambda s: s.dropna().str.len().mean()).round(1),
})
text_rank = text_rank[text_rank["filled_rows"] > 0].sort_values("filled_rows", ascending=False)
text_rank.head(25)

,filled_rows,avg_length
Field48,63027,4.2
Field71,25308,21.3
Field35,25307,3.0
Field34,25305,22.4
Field21_Text,23891,15.3
Field61,23891,19.8
Field47,23648,157.8
Field69,12978,4.6
Field79,11737,5.1
Field36,11064,146.2


## Step 7: Combine the text columns into one
Join all long-text columns into a single `text` column per post, removing repeated copies of the same text.

In [25]:
TEXT_COLS = text_rank[text_rank["avg_length"] >= 30].index.tolist()
print("Text columns used:", TEXT_COLS)

def join_text(row):
    parts = [str(v).strip() for v in row.dropna() if str(v).strip()]
    return " ".join(dict.fromkeys(parts)) if parts else None

df["text"] = df[TEXT_COLS].apply(join_text, axis=1)
print("Rows with text:", df["text"].notna().sum(), "of", len(df))

Text columns used: ['Field47', 'Field36', 'Field1', 'Field24', 'Field52', 'Field84', 'Field80', 'Field85', 'Field77', 'Field78', 'Field16', 'Field11', 'Field53', 'Field81', 'Field82', 'Field54', 'Field58_Text', 'Field65', 'Field55', 'Field66']
Rows with text: 49693 of 63031


## Step 8: Keep only the needed columns
Keep `patient`, `post_time` and `text`, and drop everything else.

In [26]:
df = df.rename(columns={"Field2": "post_time"})[["patient", "post_time", "text"]]
print("Columns now:", list(df.columns))

Columns now: ['patient', 'post_time', 'text']


## Step 9: Drop empty posts
Remove posts with no text (photos, shares, or empty rows), since there are no words to analyze.

In [27]:
before = len(df)
df["text"] = df["text"].str.strip()
df = df[df["text"].notna() & (df["text"] != "")]
funnel.append(("Posts with text", len(df), df["patient"].nunique()))
print("Rows before:", before, "| removed:", before - len(df), "| now:", len(df))

Rows before: 63031 | removed: 13338 | now: 49693


## Step 10: Drop exact duplicates
Remove posts with the same patient, text and time (scraping repeats). Real repeats on different days are kept, since repetition can be a symptom.

In [28]:
before = len(df)
df = df.drop_duplicates(subset=["patient", "text", "post_time"])
funnel.append(("No exact duplicates", len(df), df["patient"].nunique()))
print("Duplicates removed:", before - len(df), "| now:", len(df))

Duplicates removed: 1113 | now: 48580


It converts the post time into a real date.

In [29]:
import re

# ⚠️ Set this to the date the data was collected (ask your doctor if unsure)
SCRAPE_DATE = pd.Timestamp("2026-07-20")

def parse_time(s):
    if pd.isna(s):
        return pd.NaT
    s = str(s).strip()
    m = re.match(r"^(\d+)\s*(min|mins|m|hr|hrs|h|d|day|days)\b", s, re.I)
    if m:
        n, u = int(m.group(1)), m.group(2).lower()
        unit = "minutes" if u.startswith("m") else "hours" if u.startswith("h") else "days"
        return SCRAPE_DATE - pd.Timedelta(**{unit: n})
    if s.lower().startswith("yesterday"):
        return SCRAPE_DATE - pd.Timedelta(days=1)
    s = s.replace(" at ", " ")
    if not re.search(r"\b\d{4}\b", s):
        s = re.sub(r"^([A-Za-z]+ \d{1,2})", rf"\1 {SCRAPE_DATE.year}", s)
        d = pd.to_datetime(s, errors="coerce")
        if pd.notna(d) and d > SCRAPE_DATE:
            d = d - pd.DateOffset(years=1)
        return d
    return pd.to_datetime(s, errors="coerce")

unique_times = df["post_time"].dropna().unique()
time_map = {t: parse_time(t) for t in unique_times}
df["date"] = df["post_time"].map(time_map)

print("Dates parsed:", df["date"].notna().sum(), "of", len(df))
print("Not parsed:", df["date"].isna().sum())
print("Date range:", df["date"].min(), "→", df["date"].max())

Dates parsed: 48550 of 48580
Not parsed: 30
Date range: 2009-10-06 01:30:00 → 2026-07-19 23:55:00


The 30 posts without a date are too few to matter. Remove them

In [30]:
before = len(df)
df = df[df["date"].notna()]
funnel.append(("Posts with a valid date", len(df), df["patient"].nunique()))
print("Removed (no date):", before - len(df), "| now:", len(df))

Removed (no date): 30 | now: 48550


In [31]:
MIN_POSTS = 5
counts = df["patient"].value_counts()
keep = counts[counts >= MIN_POSTS].index
removed_patients = (counts < MIN_POSTS).sum()
df = df[df["patient"].isin(keep)]
funnel.append((f"Patients with ≥{MIN_POSTS} posts", len(df), df["patient"].nunique()))
print("Patients removed:", removed_patients, "| patients now:", df["patient"].nunique(), "| rows:", len(df))

Patients removed: 3 | patients now: 209 | rows: 48542


**Result:** 3 patients with fewer than 5 posts removed; 209 patients and 48,542 posts remain.

In [33]:
%pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel 


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
labels = pd.read_excel(ROOT / "data" / "raw" / "Model data (2).xlsx")
print("Rows:", len(labels), "| Columns:", labels.shape[1])
pd.DataFrame({
    "type": labels.dtypes.astype(str),
    "missing": labels.isna().sum(),
    "unique": labels.nunique(),
})

Rows: 223 | Columns: 9


,type,missing,unique
Ref. Number,float64,1,222
MOCA B Score,float64,1,15
Score Meaning,str,1,8
Unnamed: 3,float64,223,0
Unnamed: 4,float64,223,0
Unnamed: 5,float64,223,0
Unnamed: 6,float64,223,0
Unnamed: 7,float64,223,0
Unnamed: 8,str,221,2


In [35]:
labels = labels[["Ref. Number", "MOCA B Score", "Score Meaning"]].dropna(subset=["Ref. Number"])
print(labels["Score Meaning"].value_counts())
print()
print(labels["MOCA B Score"].describe().round(1))

Score Meaning
MCI (online)        127
Normal (Online)      49
Dementia(online)     28
Normal                9
Normal (online)       4
MCI(online)           2
Demensia online       2
Normal(online)        1
Name: count, dtype: int64

count    222.0
mean      16.6
std        3.1
min        7.0
25%       15.0
50%       17.0
75%       19.0
max       22.0
Name: MOCA B Score, dtype: float64


In [36]:
def group_of(meaning):
    m = str(meaning).lower()
    if "normal" in m:
        return "Normal"
    if "mci" in m:
        return "MCI"
    if "dem" in m:
        return "Dementia"
    return None

labels["group"] = labels["Score Meaning"].map(group_of)
labels["label"] = (labels["group"] != "Normal").astype(int)        # 0 = Normal, 1 = MCI/Dementia
labels["patient"] = labels["Ref. Number"].astype(int).map(lambda n: f"P{n:03d}")
labels = labels.rename(columns={"MOCA B Score": "moca"})

df = df.merge(labels[["patient", "group", "moca", "label"]], on="patient", how="left")

per_patient = df.drop_duplicates("patient")
print("Patients without a label:", per_patient["label"].isna().sum())
print(per_patient["group"].value_counts())
print(per_patient["label"].value_counts())

Patients without a label: 0
group
MCI         119
Normal       61
Dementia     29
Name: count, dtype: int64
label
1    148
0     61
Name: count, dtype: int64


In [37]:
funnel_df = pd.DataFrame(funnel, columns=["step", "rows", "patients"])
funnel_df

,step,rows,patients
0,Loaded,117474,212
1,Patient's own posts only,63031,212
2,Posts with text,49693,212
3,No exact duplicates,48580,212
4,Posts with a valid date,48550,212
5,Patients with ≥5 posts,48542,209


In [38]:
out = ROOT / "data" / "interim" / "clean_basic.csv"
df.to_csv(out, index=False, encoding="utf-8-sig")
funnel_df.to_csv(ROOT / "phase1_data" / "cleaning_funnel.csv", index=False)
print("Saved:", out.relative_to(ROOT), "| rows:", len(df), "| patients:", df["patient"].nunique())

Saved: data\interim\clean_basic.csv | rows: 48542 | patients: 209
